# Data formats

> What each object of a round contains

::: {.callout-note appearance="simple" icon=false}
**Reference: look up a fact.** It describes what trufl does, without advising or teaching. Want to know why? Read an [explanation](../explanation/03_round.ipynb). [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

In the order in which a round produces them.

| Object | Type | Index | Contents | Produced by |
|---|---|---|---|---|
| Areas | `GeoDataFrame` | `loc_id`, unique | Polygons or multipolygons | `gridder`, or `geopandas.read_file` |
| Measurements | `GeoDataFrame` | `loc_id`, repeated | `Point` geometry and a numeric `value` | `DataCollector.collect`, or field results |
| State | `DataFrame` | `loc_id` | One numeric column per criterion, one row per area | Calling a `State` |
| Ranks | `DataFrame` | `loc_id`, sorted by priority | `rank`, where 1 is first | `Optimizer.get_rank` |
| Counts | NumPy array | None: positions | Samples per area, in the order of the ranks given | `rank_to_sample` |
| Plan | `GeoDataFrame` | `loc_id` | `Point` or `MultiPoint`. Areas with no sample are absent | `Sampler.sample` |

## Coordinate systems

No function reprojects its inputs.

| Function | Requirement | Coordinates of the output |
|---|---|---|
| `gridder` | None | Those of the raster |
| `PriorCB` | Areas in the coordinates of the raster | No geometry |
| `Sampler.sample` | None | Those of the areas |
| `DataCollector.collect` | Points in the coordinates of the raster | Those of the points |
| `MoranICB` | Distances are computed in the units of the coordinates, degrees included | No geometry |

## Empty measurements

Before a first round, `State` takes an empty table:

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

In [ ]:
no_samples = gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None})
no_samples

,value
loc_id,


In [ ]:
#| hide
areas = gridder('../files/ground-truth-01-4326-simulated.tif')
n = np.zeros(len(areas), int); n[:2] = [1, 3]
plan = Sampler(areas).sample(n, rng=np.random.default_rng(0))
test_eq(len(plan), 2); test_eq(plan.index.name, 'loc_id')
test_eq(set(plan.geom_type) <= {'Point', 'MultiPoint'}, True)
test_eq(State(no_samples, areas, [CountCB()])()['Count'].isna().all(), True)